In [58]:
## Set environment variables
import os
NEO4J_URI=os.environ["NEO4J_URI"]
NEO4J_USERNAME=os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD=os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE=os.environ["NEO4J_DATABASE"]
ACADEMIC_API_KEY=os.environ["ACADEMIC_API_KEY"]

In [59]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    max_connection_lifetime=3600 
)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("RETURN 'connected' AS status")
    print(result.single())

# driver.close()


<Record status='connected'>


In [60]:
# from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph
graph=Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE, 
    sanitize=True # Cypher queries generated are safe to execute
)

In [61]:
graph

In [ ]:
# from langchain_openai import ChatOpenAI

# CYPHER_MODEL="gpt-oss-120b"
# QA_MODEL="gpt-oss-120b"

# cypher_llm = ChatOpenAI(
#     api_key=ACADEMIC_API_KEY,  
#     base_url="https://chat-ai.academiccloud.de/v1",
#     model=CYPHER_MODEL,
#     temperature=0
# )

# qa_llm=ChatOpenAI(
#     api_key=ACADEMIC_API_KEY,  
#     base_url="https://chat-ai.academiccloud.de/v1",
#     model=QA_MODEL,
#     temperature=0
# )

# # llm.invoke("How tall is the Eiffel tower?")



In [68]:
from langchain_groq import ChatGroq
groq_api_key= os.environ["GROQ_API_KEY"]
qa_llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")
cypher_llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")

## Load a dataset from Hugging Face

In [63]:
from datasets import load_dataset
from langchain_core.documents import Document

# Load dataset
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# Check split name
split_name = list(dataset.keys())[0]  # usually 'train' or 'default'
print(f"Using split: {split_name}")

# Take 1% of the data
subset = dataset[split_name].select(range(int(0.001 * len(dataset[split_name]))))

# Convert to Documents
documents = [
    Document(
        page_content=f"Question: {row['instruction']}\nResponse: {row['response']}",
        metadata={"id": idx}
    )
    for idx, row in enumerate(subset)
]

print(f"Subset size: {len(documents)}")
print(documents[:2])


Using split: train
Subset size: 26
[Document(metadata={'id': 0}, page_content="Question: question about cancelling order {{Order Number}}\nResponse: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."), Document(metadata={'id': 1}, page_content="Question: i have a question about cancelling oorder {{Order Number}}\nResponse: I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority.")]


In [69]:
for i in range(26):
    row = subset[i]
    print(f"\n--- Row {i} ---")
    print("Instruction:", row["instruction"])
    print("Response:", row["response"])



--- Row 0 ---
Instruction: question about cancelling order {{Order Number}}
Response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.

--- Row 1 ---
Instruction: i have a question about cancelling oorder {{Order Number}}
Response: I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority.

--- Row 2 ---
Instruction: i need help cancelling puchase {{Order Number}}
Response: I can sense that you're seeking assistance with canceling your purchase with the purchase number {{Order Number}}. I apologize for any inconvenience caused, and I'm here to guide you through the process.

To cancel your purchase, please 

In [70]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer=LLMGraphTransformer(llm=qa_llm)

In [71]:
graph_documents=llm_transformer.convert_to_graph_documents(documents) #convert raw text to structured knowledge--> nodes + relationships

In [72]:
graph_documents

[GraphDocument(nodes=[Node(id='Order Number', type='String', properties={}), Node(id='Customer', type='Person', properties={})], relationships=[Relationship(source=Node(id='Customer', type='Person', properties={}), target=Node(id='Order Number', type='String', properties={}), type='CANCELS_ORDER', properties={})], source=Document(metadata={'id': 0}, page_content="Question: question about cancelling order {{Order Number}}\nResponse: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.")),
 GraphDocument(nodes=[Node(id='Customer', type='Person', properties={}), Node(id='{{Order Number}}', type='Order', properties={})], relationships=[Relationship(source=Node(id='Customer', type='Person', properties={}), target=Node(id='{{Order Number}}', type='Order', properties={}), type='HAS_ORDER', properties={})], source=Document(metadata={'id

In [73]:
graph_documents[0].nodes

[Node(id='Order Number', type='String', properties={}),
 Node(id='Customer', type='Person', properties={})]

In [75]:
graph_documents[0].relationships

[Relationship(source=Node(id='Customer', type='Person', properties={}), target=Node(id='Order Number', type='String', properties={}), type='CANCELS_ORDER', properties={})]

In [ ]:
# # Convert data to CSV file, keeping only instruction and response
# import pandas as pd

# # Create DataFrame from subset
# df = pd.DataFrame(subset)

# # Keep only the two columns
# df = df[['instruction', 'response']]
# #
# # Save to CSV
# df.to_csv("data/bitext.csv", index=False)

In [ ]:
# from neo4j import GraphDatabase
# import pandas as pd

# # Connect to Neo4j
# uri = "bolt://localhost:7687"
# driver = GraphDatabase.driver(uri, auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))

# # Load CSV in Python
# df = pd.read_csv("data/bitext.csv")

# # Insert into Neo4j
# with driver.session() as session:
#     for _, row in df.iterrows():
#         session.run(
#             """
#             MERGE (q:Instruction {text: $instruction})
#             MERGE (r:Response {text: $response})
#             MERGE (q)-[:HAS_RESPONSE]->(r)
#             """,
#             instruction=row['instruction'],
#             response=row['response']
#         )

# driver.close()
# print("Data imported into Neo4j")


In [60]:
# // Load the CSV file
# LOAD CSV WITH HEADERS FROM 'data/bitext.csv' AS row

# // Create an Instruction node
# MERGE (q:Instruction {text: row.instruction})

# // Create a Response node
# MERGE (r:Response {text: row.response})

# // Link Instruction to Response
# MERGE (q)-[:HAS_RESPONSE]->(r);


In [76]:
graph.add_graph_documents(graph_documents)

In [77]:
graph

In [78]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Order {id: STRING}
Question {id: STRING}
Person {id: STRING}
Website {id: STRING}
Feature {id: STRING}
Time {id: STRING}
Phone {id: STRING}
Organization {id: STRING}
Action {id: STRING}
Phone_number {id: STRING}
Resource {id: STRING}
Contact {id: STRING}
Portal {id: STRING}
Section {id: STRING}
Url {id: STRING}
Platform {id: STRING}
Company_portal {id: STRING}
Order_interaction {id: STRING}
Support_hours {id: STRING}
Time_period {id: STRING}
Service {id: STRING}
Message {id: STRING}
Interaction {id: STRING}
Support {id: STRING}
Interface {id: STRING}
String {id: STRING}
Customer {id: STRING}
Account {id: STRING}
Online company portal {id: STRING}
Order number {id: STRING}
Website url {id: STRING}
Order interaction {id: STRING}
Customer support phone number {id: STRING}
Customer support hours {id: STRING}
Phone number {id: STRING}
Chat {id: STRING}
Company portal {id: STRING}
Support hours {id: STRING}
Process {id: STRING}
Tab {id: STRING}
Order-number {id: STRING}
Onli

In [ ]:
# from langchain_core.prompts import PromptTemplate

# CYPHER_GENERATION_TEMPLATE = """
# You are a Neo4j expert.

# Generate ONLY a valid Cypher query.
# Do NOT include explanations.
# Do NOT include reasoning.
# Do NOT include <think> tags.
# Return ONLY executable Cypher.
# Use literal IDs that exist in the database
# Do NOT use placeholders like 'User', 'Order Number', '{{Order Number}}'

# Schema:
# {schema}

# Question:
# {question}
# """

# cypher_prompt = PromptTemplate(
#     input_variables=["schema", "question"],
#     template=CYPHER_GENERATION_TEMPLATE,
# )


In [ ]:
# from langchain_neo4j import GraphCypherQAChain, Neo4jGraph


# # Create the Cypher QA chain from your LLM and existing graph
# cypher_qa = GraphCypherQAChain.from_llm(
#     cypher_llm=cypher_llm,
#     qa_llm=qa_llm,
#     graph=graph,
#     # cypher_prompt=cypher_prompt,
#     verbose=True,                     # Optionally log generated Cypher
#     allow_dangerous_requests=True     # Allows execution of queries with operations like DELETE
# )

# # Run a natural language query (use existing `question` variable if present)
# nl_query = globals().get("question", "i dont know what to do to cancel order")

# # Execute and handle missing-parameter errors (e.g. missing $personId)
# try:
#     result = cypher_qa.invoke({"query": nl_query})
#     print("CypherQA result:", result)
# except Exception as e:
#     msg = str(e)
#     # Detect the ParameterMissing / personId situation and use a safe fallback query
#     if "personId" in msg or "ParameterMissing" in msg:
#         print("Generated Cypher expected a missing parameter (personId). Running safe fallback Cypher instead.")
#         fallback_cypher = """
#         MATCH (o:Order)-[:CAN_BE_CANCELED_THROUGH|:INITIATE_CANCELLATION|:HAS_CANCELLATION_POLICY]->(target)
#         RETURN DISTINCT labels(target)[0] AS label, coalesce(target.text, target.id, '') AS value
#         LIMIT 25
#         """
#         with driver.session() as session:
#             rows = list(session.run(fallback_cypher).data())
#         print("Fallback results:", rows)
#         result = rows
#     else:
#         # re-raise unexpected exceptions
#         raise




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:HAS_ORDER]->(o:Order)
OPTIONAL MATCH (o)-[:CANCEL]->(sec:Section)
OPTIONAL MATCH (o)-[:CANCEL]->(svc:Service)
OPTIONAL MATCH (o)-[:CANCELED_VIA]->(act:Action)
RETURN o.id AS orderId,
       collect(DISTINCT sec.id) AS cancelSectionIds,
       collect(DISTINCT svc.id) AS cancelServiceIds,
       collect(DISTINCT act.id) AS cancelActionIds;
Full Context:
[{'orderId': 'Order Number', 'cancelSectionIds': ['Online Order Interaction - Cancel Option'], 'cancelServiceIds': [], 'cancelActionIds': []}, {'orderId': '{{Order Number}}', 'cancelSectionIds': [], 'cancelServiceIds': ['{{Online Company Portal Info}}'], 'cancelActionIds': []}]

> Finished chain.
CypherQA result: {'query': 'How can I cancell my oeder?', 'result': "I don't know the answer."}


In [79]:
# GraphCypherQAChain comes from langchain_neo4j; use the existing qa_llm and cypher_llm
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
	llm=qa_llm,
	cypher_llm=cypher_llm,
	graph=graph,
    # cypher_prompt=cypher_prompt,
	verbose=True,
    allow_dangerous_requests=True
)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x0000013C0323CA50>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False,

## Q1

In [22]:
response=chain.invoke({"query":"i want assistance cancelling order"})
response





> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:CANCEL_ORDER_INQUIRY|REQUEST_CANCEL|CANCEL|INITIATE_CANCELLATION]->(o:Order)
RETURN p.id AS personId, o.id AS orderId
Full Context:
[{'personId': 'User', 'orderId': '{{Order Number}}'}, {'personId': 'User', 'orderId': '{{Order Number}}'}, {'personId': 'User', 'orderId': '{{Order Number}}'}, {'personId': 'Customer', 'orderId': '{{Order Number}}'}, {'personId': 'User', 'orderId': '{{Order Number}}'}, {'personId': 'User', 'orderId': 'Order Number'}, {'personId': 'User', 'orderId': 'Order {{Order Number}}'}]

> Finished chain.


{'query': 'i want assistance cancelling order',
 'result': 'I see you have several orders listed (e.g.,\u202f{{Order Number}}). Please let me know which specific order you’d like to cancel, and I’ll take care of it for you.'}

In [28]:
response=chain.invoke({"query":"i want assistance cancelling order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[r:CANCEL_ORDER_INQUIRY|REQUEST_CANCEL|CANCEL|CANCEL_REQUEST|INITIATE_CANCELLATION]->(o:Order)
RETURN p.id AS personId, o.id AS orderId, type(r) AS relType;
Full Context:
[{'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'CANCEL_ORDER_INQUIRY'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'REQUEST_CANCEL'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'CANCEL'}, {'personId': 'Customer', 'orderId': '{{Order Number}}', 'relType': 'CANCEL'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'CANCEL_REQUEST'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'INITIATE_CANCELLATION'}, {'personId': 'User', 'orderId': 'Order Number', 'relType': 'CANCEL'}, {'personId': 'User', 'orderId': 'Order Number', 'relType': 'CANCEL_REQUEST'}, {'personId': 'User', 'orderId': 'Order {{Order Number}}', 'relType': 'REQUEST_CANCEL'}]

> Finished chain.


{'query': 'i want assistance cancelling order',
 'result': 'You can cancel the order by using one of the following actions:\u202fCANCEL_ORDER_INQUIRY,\u202fREQUEST_CANCEL,\u202fCANCEL,\u202fCANCEL_REQUEST,\u202for\u202fINITIATE_CANCELLATION.'}

In [30]:
response=chain.invoke({"query":"How can I cancel my order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[r:CANCEL|REQUEST_CANCEL|CANCEL_ORDER_INQUIRY|CANCEL_REQUEST|INITIATE_CANCELLATION]->(o:Order)
RETURN p.id AS personId, o.id AS orderId, type(r) AS relType;
Full Context:
[{'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'CANCEL_ORDER_INQUIRY'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'REQUEST_CANCEL'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'CANCEL'}, {'personId': 'Customer', 'orderId': '{{Order Number}}', 'relType': 'CANCEL'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'CANCEL_REQUEST'}, {'personId': 'User', 'orderId': '{{Order Number}}', 'relType': 'INITIATE_CANCELLATION'}, {'personId': 'User', 'orderId': 'Order Number', 'relType': 'CANCEL'}, {'personId': 'User', 'orderId': 'Order Number', 'relType': 'CANCEL_REQUEST'}, {'personId': 'User', 'orderId': 'Order {{Order Number}}', 'relType': 'REQUEST_CANCEL'}]

> Finished chain.


{'query': 'How can I cancel my order',
 'result': 'You can cancel your order by submitting a cancellation request—e.g., use the “request cancel” or “cancel request” action, or initiate the cancellation directly. Once the request is received, the order will be marked as cancelled.'}

## Q2

In [31]:
response=chain.invoke({"query":"How  to place an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ASK]->(o:Order)
RETURN p.id AS personId, o.id AS orderId;
Full Context:
[{'personId': 'Customer', 'orderId': '{{Order Number}}'}]

> Finished chain.


{'query': 'How  to place an order',
 'result': 'The customer places an order by providing the order number\u202f{{Order Number}}.'}

In [31]:
response=chain.invoke({"query":"How  to place an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:NAVIGATE|INITIATE]->(f:Feature)-[:MANAGES]->(o:Order)
RETURN p.id AS personId, f.id AS featureId, o.id AS orderId;
Full Context:
[]

> Finished chain.


{'query': 'How  to place an order', 'result': 'I don’t know the answer.'}

In [32]:
response=chain.invoke({"query":"i need assisstance placing an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Order)<-[:SUPPORT_FOR]-(w:Website)
OPTIONAL MATCH (p:Person)-[:CONTACT_SUPPORT]->(w)
RETURN o.id AS orderId, w.id AS websiteId, collect(p.id) AS supportPersons;
Full Context:
[{'orderId': '{{Order Number}}', 'websiteId': '{{Website Url}}', 'supportPersons': ['User']}]

> Finished chain.


{'query': 'i need assisstance placing an order',
 'result': 'You can place your order on {{Website Url}} using order number {{Order Number}}. If you need help, you can contact User for assistance.'}

In [36]:
response=chain.invoke({"query":"i need some help to place an order"})
response




> Entering new GraphCypherQAChain chain...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. ':MANAGE|:APPLY' is deprecated. It is replaced by ':MANAGE|APPLY'.", position=<SummaryInputPosition line=2, column=35, offset=99>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 99, 'line': 2, 'column': 35}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (w:Website)-[:PROVIDES]->(f:Feature)-[:MANAGES]->(o:Order)\nOPTIONAL MATCH (a:Action)-[:MANAGE|:APPLY]->(o)\nRETURN w.id AS website, f.id AS feature, a.id AS action\nLIMIT 10'


Generated Cypher:
MATCH (w:Website)-[:PROVIDES]->(f:Feature)-[:MANAGES]->(o:Order)
OPTIONAL MATCH (a:Action)-[:MANAGE|:APPLY]->(o)
RETURN w.id AS website, f.id AS feature, a.id AS action
LIMIT 10
Full Context:
[{'website': 'Online Company Portal Info', 'feature': 'Online Order Interaction', 'action': None}]

> Finished chain.


{'query': 'i need some help to place an order',
 'result': 'You can place your order through the Online Company Portal, which provides an online order interaction feature.'}

In [47]:
response=chain.invoke({"query":" How can i place an order"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "some_person_id"})-[:INITIATE]->(f:Feature)-[:MANAGES]->(o:Order)
RETURN o.id AS orderId
UNION
MATCH (p:Person {id: "some_person_id"})-[:INITIATE]->(a:Action)-[:MANAGE]->(o:Order)
RETURN o.id AS orderId
Full Context:
[]

> Finished chain.


{'query': ' How can i place an order', 'result': 'I don’t know the answer.'}

## Q3

In [46]:
response=chain.invoke({"query":" I am waiting for an refund of {{Currency Symbol}}{{Refund Amount}}"})
response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:HAS_PROBLEM_WITH]->(o:Order)
RETURN p.id AS personId, o.id AS orderId
Full Context:
[{'personId': 'User', 'orderId': '{{Order Number}}'}]

> Finished chain.


{'query': ' I am waiting for an refund of {{Currency Symbol}}{{Refund Amount}}',
 'result': 'I don’t know the answer.'}

In [51]:
response=chain.invoke({"query":" want assistance to know if there are any news on my rebate"})
response





> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Message)
WHERE m.id CONTAINS 'rebate'
RETURN m;
Full Context:
[]

> Finished chain.


{'query': ' want assistance to know if there are any news on my rebate',
 'result': 'I’m sorry, but I don’t have the information needed to answer that.'}

In [52]:
response=chain.invoke({"query":" see the status of the reimbursement"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Order) RETURN o.id;
Full Context:
[{'o.id': 'Order'}, {'o.id': '{{Order Number}}'}, {'o.id': 'Order Number'}, {'o.id': 'Order {{Order Number}}'}]

> Finished chain.


{'query': ' see the status of the reimbursement',
 'result': 'I don’t know the answer.'}

### BERTScore

## BERTScore

In [80]:
import json
from tqdm import tqdm
from bert_score import score

def evaluate_graph_rag_with_bertscore(chain, subset, max_examples=None):
    """
    Runs GraphCypherQAChain on test examples and computes BERTScore.
    
    Args:
        chain: GraphCypherQAChain instance
        subset: HuggingFace dataset subset with fields ['instruction', 'response']
        max_examples: int | None (limit evaluation size for faster runs)
        
    Returns:
        avg_precision, avg_recall, avg_f1
    """

    references = []
    predictions = []

    examples = subset
    if max_examples is not None:
        examples = subset.select(range(min(max_examples, len(subset))))

    for row in tqdm(examples, desc="Evaluating Graph RAG"):
        question = row["instruction"]
        reference = row["response"]

        try:
            result = chain.invoke({"query": question})

            # LangChain GraphCypherQAChain returns dict like:
            # {"query": "...", "result": "..."}
            predicted_answer = result.get("result", "")

        except Exception as e:
            print(f"[WARN] Query failed: {question[:80]}...")
            print(e)
            predicted_answer = ""

        predictions.append(predicted_answer)
        references.append(reference)

    # Compute BERTScore
    P, R, F1 = score(
        predictions,
        references,
        lang="en",
        model_type="distilroberta-base",
        batch_size=16,
        rescale_with_baseline=True
    )

    avg_precision = P.mean().item()
    avg_recall = R.mean().item()
    avg_f1 = F1.mean().item()

    print(f"\n[RESULT] BERTScore")
    print(f"Precision: {avg_precision:.4f}")
    print(f"Recall   : {avg_recall:.4f}")
    print(f"F1       : {avg_f1:.4f}")

    return avg_precision, avg_recall, avg_f1


In [81]:
# Evaluate on the same subset inserted into Neo4j
evaluate_graph_rag_with_bertscore(
    chain=chain,
    subset=subset,
    max_examples=30  
)

Evaluating Graph RAG:   0%|          | 0/26 [00:00<?, ?it/s]



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (o:Order {id: "{{Order Number}}"}),(p:Person)-[:CONFIRM|CANCEL|LOCATE_IN_SECTION|CLICK|GO_TO|CONTACT]->(f:Feature)-[:PROVIDES]->(s:Service)-[:OFFER]->(o) RETURN p.id AS user_id, f.id AS feature_id;
Full Context:
[]


Evaluating Graph RAG:   4%|▍         | 1/26 [00:01<00:31,  1.28s/it]


> Finished chain.


> Entering new GraphCypherQAChain chain...


Evaluating Graph RAG:   8%|▊         | 2/26 [00:07<01:35,  4.00s/it]

...


> Finished chain.


> Entering new GraphCypherQAChain chain...


Evaluating Graph RAG: 100%|██████████| 26/26 [12:54<00:00, 29.78s/it]

Generated Cypher:
MATCH (p:Person)-[:INITIATE_CANCELLATION]->(o:Order {id: $OrderNumber}) RETURN p
[WARN] Query failed: i want assistance cancelling order {{Order Number}}...
{neo4j_code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): OrderNumber} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 284.38it/s, Materializing param=pooler.dense.weight]                             
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: RobertaTokenizer has no attribute build_inputs_with_special_tokens